# Setup for Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
ROOT = "/content/drive/MyDrive/songs"

In [ ]:
import os

songs = []

for folder in os.listdir(ROOT):
    folder_path = os.path.join(ROOT, folder)

    if not os.path.isdir(folder_path):
        continue

    for file in os.listdir(folder_path):
        if file.lower().endswith((".mp3", ".wav", ".m4a", ".flac")):
            songs.append({
                "folder": folder,
                "path": os.path.join(folder_path, file),
                "file": file
            })

print("Total songs:", len(songs))
print(songs[:5])

Total songs: 121
[{'folder': 'pluggnb', 'path': '/content/drive/MyDrive/songs/pluggnb/Hina, Vlonezy - Aaliyah (SPOTISAVER).mp3', 'file': 'Hina, Vlonezy - Aaliyah (SPOTISAVER).mp3'}, {'folder': 'pluggnb', 'path': '/content/drive/MyDrive/songs/pluggnb/jssr - rue (SPOTISAVER).mp3', 'file': 'jssr - rue (SPOTISAVER).mp3'}, {'folder': 'pluggnb', 'path': '/content/drive/MyDrive/songs/pluggnb/drown!, jssr - asian (SPOTISAVER).mp3', 'file': 'drown!, jssr - asian (SPOTISAVER).mp3'}, {'folder': 'pluggnb', 'path': '/content/drive/MyDrive/songs/pluggnb/Baby Boat - Abu (SPOTISAVER).mp3', 'file': 'Baby Boat - Abu (SPOTISAVER).mp3'}, {'folder': 'pluggnb', 'path': '/content/drive/MyDrive/songs/pluggnb/jssr - company (SPOTISAVER).mp3', 'file': 'jssr - company (SPOTISAVER).mp3'}]


# Song Embeddings and Plot
### Each song is an embedding, and is plotted on a 3d field

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import librosa
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px

In [ ]:
from pathlib import Path
import numpy as np
import librosa
ROOT = Path("/content/drive/MyDrive/songs")
AUDIO_EXTS = [".mp3", ".wav", ".flac", ".m4a"]

# ---- find all songs ----
song_paths = [
    p for p in ROOT.rglob("*")
    if p.suffix.lower() in AUDIO_EXTS
]

print(f"Found {len(song_paths)} songs")

# ---- containers ----
embeddings = []
names = []
folders = []

# ---- embed each song ----
for path in song_paths:
    print(f"Embedding {path.name}")

    y, sr = librosa.load(
        path,
        sr=22050,
        mono=True
    )

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=40
    )

    # Chroma
    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr
    )

    # Spectral contrast
    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr
    )

    # combine features
    vec = np.concatenate([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),

        chroma.mean(axis=1),
        chroma.std(axis=1),

        contrast.mean(axis=1),
        contrast.std(axis=1),
    ])

    embeddings.append(vec)
    names.append(path.stem)
    folders.append(path.parent.name)

# ---- convert to arrays ----
embeddings = np.array(embeddings)
names = np.array(names)
folders = np.array(folders)

# ---- sanity check ----
print("Embeddings shape:", embeddings.shape)
print("Example song:", names[0])

# ---- optional save ----
np.save("embeddings.npy", embeddings)
np.save("song_names.npy", names)
np.save("song_folders.npy", folders)

print("Saved embeddings.npy / song_names.npy / song_folders.npy")

Found 121 songs
Embedding Hina, Vlonezy - Aaliyah (SPOTISAVER).mp3
Embedding jssr - rue (SPOTISAVER).mp3
Embedding drown!, jssr - asian (SPOTISAVER).mp3
Embedding Baby Boat - Abu (SPOTISAVER).mp3
Embedding jssr - company (SPOTISAVER).mp3
Embedding jssr - cell (SPOTISAVER).mp3
Embedding Lil Shine - Like What (SPOTISAVER).mp3
Embedding jssr, Corey Lingo - minute (SPOTISAVER).mp3
Embedding feel_spotdown.org.mp3
Embedding west_spotdown.org.mp3
Embedding bout it_spotdown.org.mp3
Embedding i can't feel at all_spotdown.org.mp3
Embedding heart_spotdown.org.mp3
Embedding angel_spotdown.org.mp3
Embedding drunk_spotdown.org.mp3
Embedding thinking_spotdown.org.mp3
Embedding painting_spotdown.org.mp3
Embedding love_spotdown.org.mp3
Embedding history_spotdown.org.mp3
Embedding Choose Up (feat. Corey Lingo)_spotdown.org.mp3
Embedding funny, c4pri - push (SPOTISAVER).mp3
Embedding gicc, c4pri - worst (SPOTISAVER).mp3
Embedding 1crusafix, isai, akkiro - problem (SPOTISAVER).mp3
Embedding prkr blu, 1cru

In [ ]:
coords = PCA(n_components=3).fit_transform(embeddings)

df = pd.DataFrame({
    "song": names,
    "folder": folders,
    "x": coords[:, 0],
    "y": coords[:, 1],
    "z": coords[:, 2],
})

fig = px.scatter_3d(
    df,
    x="x",
    y="y",
    z="z",
    color="folder",
    hover_name="song",
    title="Song Embedding Space"
)

fig.show()

# With this, we can see that music pertaining to genres have some sort of cluster. This implies that songs next to each other have similarity. Hence, why not use nearest neighbors to songs as particular candidates

In [51]:
sim = cosine_similarity(embeddings)

song_to_idx = {name: i for i, name in enumerate(names)}

def nearest_neighbors(song_name, k=10):
    idx = song_to_idx[song_name]
    scores = sim[idx]
    order = np.argsort(scores)[::-1]

    results = []
    for j in order:
        if j == idx:
            continue

        results.append({
            "song": names[j],
            "folder": folders[j],
            "score": float(scores[j])
        })

        if len(results) >= k:
            break

    return pd.DataFrame(results)

In [52]:
i = 90
print(names[i])
nearest_neighbors(names[i],k=5)

Lift, Jump, Exhale_spotdown.org


,song,folder,score
0,À QUI DE DROIT_spotdown.org,Fete-Aesthetic,0.978537
1,1tap_spotdown.org,hyperpop,0.977885
2,Lucy Bedroque - INFINITUDE UROBOROS (SPOTISAVER),Fete-Aesthetic,0.976975
3,Magic Johnson_spotdown.org,rap,0.975063
4,Lucy Bedroque - Knot Me (SPOTISAVER),Fete-Aesthetic,0.975020


## Embedding from Retreival

In [50]:
room = {
    "user_A": [names[0], names[1], names[2]],
    "user_B": [names[10], names[11], names[12]],
    "user_C": [names[20], names[21], names[22]],
}

In [ ]:
print(room)

{'user_A': [np.str_('Hina, Vlonezy - Aaliyah (SPOTISAVER)'), np.str_('jssr - rue (SPOTISAVER)'), np.str_('drown!, jssr - asian (SPOTISAVER)')], 'user_B': [np.str_('bout it_spotdown.org'), np.str_("i can't feel at all_spotdown.org"), np.str_('heart_spotdown.org')], 'user_C': [np.str_('funny, c4pri - push (SPOTISAVER)'), np.str_('gicc, c4pri - worst (SPOTISAVER)'), np.str_('1crusafix, isai, akkiro - problem (SPOTISAVER)')]}


Retrieval scoring:

For every user in the room:
  Look at that user’s liked songs.

For each liked song:
  Find its n-nearest neighbors in embedding space.

For each retrieved neighbor:
  Add it to the candidate pool.

But:
  If the same user retrieves the same candidate multiple times,
  only keep that user's strongest similarity score for that candidate.

So each candidate stores:
  candidate_song -> { user -> best similarity }

Example:
  Song X -> {
    User A: 0.91,
    User B: 0.76
  }

This means:
  Song X was surfaced by User A and User B,
  and those are the strongest similarities from each user’s liked songs.

Then retrieval score can be:
  average similarity among users who surfaced it
  times
  fraction of room users who surfaced it.

After retrieval:
  We compute the true room score by scoring the candidate against all users,
  not only the users who retrieved it.
  

In [ ]:
def retrieve_candidates(room, neighbors_per_seed=10):
    candidates = {}

    for user, liked_songs in room.items():
        for song in liked_songs:
            neighbors = nearest_neighbors(song, k=neighbors_per_seed)

            for _, row in neighbors.iterrows():
                candidate = row["song"]
                sim = row["score"]

                if candidate not in candidates:
                    candidates[candidate] = {}

                # keep best similarity for this user
                prev = candidates[candidate].get(user, -1)
                candidates[candidate][user] = max(prev, sim)

            # include liked song too
            if song not in candidates:
                candidates[song] = {}

            candidates[song][user] = max(
                candidates[song].get(user, -1),
                1.0
            )

    return candidates

In [ ]:
retrieved = retrieve_candidates(room, neighbors_per_seed=10)

rows = []

for song, user_sims in retrieved.items():
    retrieval_score = sum(user_sims.values())
    user_coverage = len(user_sims)

    rows.append({
        "song": song,
        "folder": folders[song_to_idx[song]],
        "retrieval_score": retrieval_score,
        "user_coverage": user_coverage,
        "users_retrieved": list(user_sims.keys()),
        "retrieval_details": user_sims,
    })

retrieval_df = (
    pd.DataFrame(rows)
    .sort_values("retrieval_score", ascending=False)
)

retrieval_df.head(20)

,song,folder,retrieval_score,user_coverage,users_retrieved,retrieval_details
4,love_spotdown.org,pluggnb,1.980519,2,"[user_A, user_B]","{'user_A': 0.9805192839805914, 'user_B': 1.000..."
10,"Hina, Vlonezy - Aaliyah (SPOTISAVER)",pluggnb,1.980519,2,"[user_A, user_B]","{'user_A': 1.0, 'user_B': 0.9805192839805914}"
5,heart_spotdown.org,pluggnb,1.980519,2,"[user_A, user_B]","{'user_A': 0.9805192839805914, 'user_B': 1.0}"
20,jssr - rue (SPOTISAVER),pluggnb,1.977057,2,"[user_A, user_B]","{'user_A': 1.0, 'user_B': 0.9770569081207116}"
13,i can't feel at all_spotdown.org,pluggnb,1.977057,2,"[user_A, user_B]","{'user_A': 0.9770569081207116, 'user_B': 1.0}"
1,jssr - cell (SPOTISAVER),pluggnb,1.967675,2,"[user_A, user_B]","{'user_A': 0.985577863136098, 'user_B': 0.9820..."
0,Baby Boat - Abu (SPOTISAVER),pluggnb,1.967436,2,"[user_A, user_B]","{'user_A': 0.9881959276300137, 'user_B': 0.979..."
6,jssr - company (SPOTISAVER),pluggnb,1.966931,2,"[user_A, user_B]","{'user_A': 0.9790430621194287, 'user_B': 0.987..."
2,5g_spotdown.org,hyperpop,1.966883,2,"[user_A, user_B]","{'user_A': 0.9823822745070918, 'user_B': 0.984..."
3,thinking_spotdown.org,pluggnb,1.966204,2,"[user_A, user_B]","{'user_A': 0.9822246553027066, 'user_B': 0.983..."


In [ ]:
top_candidates = retrieval_df.head(40)["song"].tolist()

In [ ]:
print(top_candidates)

[np.str_('love_spotdown.org'), np.str_('Hina, Vlonezy - Aaliyah (SPOTISAVER)'), np.str_('heart_spotdown.org'), np.str_('jssr - rue (SPOTISAVER)'), np.str_("i can't feel at all_spotdown.org"), np.str_('jssr - cell (SPOTISAVER)'), np.str_('Baby Boat - Abu (SPOTISAVER)'), np.str_('jssr - company (SPOTISAVER)'), np.str_('5g_spotdown.org'), np.str_('thinking_spotdown.org'), np.str_('KELLY KELLY_spotdown.org'), np.str_('Sunflower - Spider-Man_ Into the Spider-Verse_spotdown.org'), np.str_('Bad Liar_spotdown.org'), np.str_('Pretty Peach_spotdown.org'), np.str_('kuru - Noir kei (SPOTISAVER)'), np.str_('AFFLICTED - firstplace (SPOTISAVER)'), np.str_('drown!, jssr - asian (SPOTISAVER)'), np.str_('1crusafix, isai, akkiro - problem (SPOTISAVER)'), np.str_('gicc, c4pri - worst (SPOTISAVER)'), np.str_('funny, c4pri - push (SPOTISAVER)'), np.str_('bout it_spotdown.org'), np.str_('prkr blu, 1crusafix, trustt - #SILKROADD (SPOTISAVER)'), np.str_('history_spotdown.org'), np.str_('Temperature, any other 

In [ ]:
def user_score(user_liked_songs, candidate_song, top_k=2):
    candidate_idx = song_to_idx[candidate_song]

    scores = []
    for liked_song in user_liked_songs:
        liked_idx = song_to_idx[liked_song]
        scores.append(sim[candidate_idx, liked_idx])

    scores = sorted(scores, reverse=True)
    return float(np.mean(scores[:top_k]))


def room_score(room, candidate_song, top_k=2):
    per_user = {}

    for user, liked_songs in room.items():
        per_user[user] = user_score(liked_songs, candidate_song, top_k=top_k)

    avg = np.mean(list(per_user.values()))

    return avg, per_user

In [ ]:
rows = []

for song in top_candidates:
    score, per_user = room_score(room, song, top_k=2)

    rows.append({
        "song": song,
        "folder": folders[song_to_idx[song]],
        "room_score": score,
        **per_user
    })

ranked_df = (
    pd.DataFrame(rows)
    .sort_values("room_score", ascending=False)
)

ranked_df.head(40)

,song,folder,room_score,user_A,user_B,user_C
38,STAY (with Justin Bieber)_spotdown.org,pop,0.950343,0.961807,0.953450,0.935772
0,love_spotdown.org,pluggnb,0.948311,0.971342,0.992648,0.880942
2,heart_spotdown.org,pluggnb,0.948311,0.971342,0.992648,0.880942
11,Sunflower - Spider-Man_ Into the Spider-Verse_...,pop,0.948272,0.970119,0.972649,0.902048
35,Say So_spotdown.org,pop,0.947346,0.967832,0.956395,0.917810
23,"Temperature, any other way 2002_spotdown.org",Fete-Aesthetic,0.947013,0.967158,0.978621,0.895262
27,"thank u, next_spotdown.org",pop,0.946775,0.967276,0.965327,0.907723
7,jssr - company (SPOTISAVER),pluggnb,0.945438,0.972970,0.985835,0.877508
31,Inside My Heart_spotdown.org,Fete-Aesthetic,0.941500,0.972603,0.959599,0.892298
1,"Hina, Vlonezy - Aaliyah (SPOTISAVER)",pluggnb,0.941486,0.978011,0.976743,0.869704


In [ ]:
def build_queue(ranked_df, queue_len=5):
    queue = []
    used = set()

    for _ in range(queue_len):
        best_song = None
        best_score = -float("inf")

        for _, row in ranked_df.iterrows():
            song = row["song"]

            if song in used:
                continue

            score = row["room_score"]

            # simple anti-repetition by folder
            if queue:
                prev_song = queue[-1]
                prev_folder = folders[song_to_idx[prev_song]]

                if row["folder"] == prev_folder:
                    score -= 0.10

            if score > best_score:
                best_score = score
                best_song = song

        queue.append(best_song)
        used.add(best_song)

    return queue

queue = build_queue(ranked_df, queue_len=5)
queue

[np.str_('STAY (with Justin Bieber)_spotdown.org'),
 np.str_('love_spotdown.org'),
 np.str_('Sunflower - Spider-Man_ Into the Spider-Verse_spotdown.org'),
 np.str_('heart_spotdown.org'),
 np.str_('Say So_spotdown.org')]

In [ ]:
def transition_score(prev_song, next_song):
    i = song_to_idx[prev_song]
    j = song_to_idx[next_song]

    # cosine similarity from embeddings
    return float(sim[i, j])

In [ ]:
transition_score(names[0],names[40])

0.9157991181197698

In [ ]:
def next_song_score(prev_song, candidate_song, room_score_value):
    t = transition_score(prev_song, candidate_song)

    return (
        room_score_value
        + 0.25 * t
    )

In [53]:
for i in names:
  for j in names:
    if(transition_score(i,j)<=0.7):
      print(str(i) + ", " + str(j) + ": " + str(transition_score(i,j)))

Hina, Vlonezy - Aaliyah (SPOTISAVER), yeil - 꿈안에서 (SPOTISAVER): 0.6878676542797445
Hina, Vlonezy - Aaliyah (SPOTISAVER), Che - DIOR LEOPARD (SPOTISAVER): 0.6899722678649067
Hina, Vlonezy - Aaliyah (SPOTISAVER), DIE HARD_spotdown.org: 0.5849436009124842
Hina, Vlonezy - Aaliyah (SPOTISAVER), KING OF ROCK_spotdown.org: 0.3809635287894911
Hina, Vlonezy - Aaliyah (SPOTISAVER), My Chemical Romance - House of Wolves (SPOTISAVER): 0.6614636755840619
Hina, Vlonezy - Aaliyah (SPOTISAVER), American Idiot_spotdown.org: 0.6790406996715306
Hina, Vlonezy - Aaliyah (SPOTISAVER), I'm Not Okay (I Promise)_spotdown.org: 0.5978548982117591
jssr - rue (SPOTISAVER), Lil Shine - Like What (SPOTISAVER): 0.6351197189633647
jssr - rue (SPOTISAVER), Che - GET NAKED (SPOTISAVER): 0.6275846131206568
jssr - rue (SPOTISAVER), Che - DIOR LEOPARD (SPOTISAVER): 0.6563931786329604
jssr - rue (SPOTISAVER), UDG Records, Africakid - Chainsaw (SPOTISAVER): 0.6584057271750273
jssr - rue (SPOTISAVER), DIE HARD_spotdown.

In [58]:
import pulp

def solve_queue_ip(
    ranked_df,
    queue_len=5,
    candidate_limit=10,
    transition_weight=0.25
):
    candidate_df = ranked_df.head(candidate_limit).copy().reset_index(drop=True)

    songs = candidate_df["song"].tolist()
    n = len(songs)
    slots = list(range(queue_len))

    room_scores = candidate_df["room_score"].to_numpy()

    # precompute transition matrix once
    trans = {}
    for a in range(n):
        for b in range(n):
            if a != b:
                trans[(a, b)] = transition_score(songs[a], songs[b])

    model = pulp.LpProblem("music_queue", pulp.LpMaximize)

    # x[a,t] = song index a placed in slot t
    x = pulp.LpVariable.dicts(
        "x",
        ((a, t) for a in range(n) for t in slots),
        cat="Binary"
    )

    # y[a,b,t] = song a in slot t followed by song b in slot t+1
    y = pulp.LpVariable.dicts(
        "y",
        ((a, b, t) for a in range(n) for b in range(n) for t in slots[:-1] if a != b),
        cat="Binary"
    )

    model += (
        pulp.lpSum(
            room_scores[a] * x[(a, t)]
            for a in range(n)
            for t in slots
        )
        +
        transition_weight * pulp.lpSum(
            trans[(a, b)] * y[(a, b, t)]
            for a in range(n)
            for b in range(n)
            for t in slots[:-1]
            if a != b
        )
    )

    # one song per slot
    for t in slots:
        model += pulp.lpSum(x[(a, t)] for a in range(n)) == 1

    # each song used at most once
    for a in range(n):
        model += pulp.lpSum(x[(a, t)] for t in slots) <= 1

    # link y = x[a,t] AND x[b,t+1]
    for a in range(n):
        for b in range(n):
            if a == b:
                continue
            for t in slots[:-1]:
                model += y[(a, b, t)] <= x[(a, t)]
                model += y[(a, b, t)] <= x[(b, t + 1)]
                model += y[(a, b, t)] >= x[(a, t)] + x[(b, t + 1)] - 1

    status = model.solve(pulp.PULP_CBC_CMD(msg=True, timeLimit=10))

    print("Status:", pulp.LpStatus[status])

    queue = []

    for t in slots:
        for a in range(n):
            val = pulp.value(x[(a, t)])
            if val is not None and val > 0.5:
                queue.append(songs[a])

    return queue

In [60]:
ip_queue = solve_queue_ip(
    ranked_df,
    queue_len=5,
    candidate_limit=10
)

print(ip_queue)

Status: Optimal
[np.str_('Hina, Vlonezy - Aaliyah (SPOTISAVER)'), np.str_('Say So_spotdown.org'), np.str_('Sunflower - Spider-Man_ Into the Spider-Verse_spotdown.org'), np.str_('Inside My Heart_spotdown.org'), np.str_('thank u, next_spotdown.org')]
